In [1]:
import jax 
import jax.numpy as jnp

from probjax.core.custom_primitives.custom_inverse import custom_inverse
from functools import partial

@partial(custom_inverse, static_argnums=(1,))
def f(x, y):
    return x**y

f.definv(lambda y, x: y**(1./x))

x = jnp.array([1., 2., 3.])
y = f(x, 2.)

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [2]:
jaxpr = jax.make_jaxpr(lambda x: f(x, 2.))(x)

In [3]:
jaxpr_forward = jaxpr.eqns[0].params["forward_jaxpr"]

jax.core.eval_jaxpr(jaxpr_forward.jaxpr, jaxpr_forward.literals, x)

[Array([1., 4., 9.], dtype=float32)]

In [4]:
inverse_forward = jaxpr.eqns[0].params["inverse_jaxpr"]

jax.core.eval_jaxpr(inverse_forward.jaxpr, inverse_forward.literals, y)

[Array([1., 2., 3.], dtype=float32), nan]

In [5]:
from probjax.utils.odeint import odeint, _odeint

drift = lambda t, x: -0.1*x
ys = odeint(drift, 1., jnp.linspace(0., 1., 100))

odeint.inv(drift, ys, jnp.linspace(0., 1., 100))
odeint.inv_and_logdet(drift, ys, jnp.linspace(0., 1., 100))

(100,)


(Array(1., dtype=float32), Array(0.1, dtype=float32))

In [9]:
from probjax.utils.odeint import odeint, _odeint
from probjax.core import inverse, inverse_and_logabsdet

ts = jnp.linspace(0., 1., 100)

def f(x):
    ys = odeint(lambda x, t: jnp.sin(x), x, ts, method="rk4")
    return ys[-1]


jaxpr = jax.make_jaxpr(f)(1.)

forward_jaxpr = jaxpr.eqns[0].params["forward_jaxpr"]
inverse_jaxpr = jaxpr.eqns[0].params["inverse_jaxpr"]

ys = f(1.)

In [10]:
inverse_and_logabsdet(f)(ys)

(Array(0.996, dtype=float32), Array(0., dtype=float32))

In [ ]:
jax.lax.dynamic_update_slice

Signature:
jax.lax.dynamic_update_slice(
    operand: Union[jax.Array, numpy.ndarray],
    update: Union[jax.Array, numpy.ndarray, numpy.bool_, numpy.number, bool, int, float, complex],
    start_indices: Union[jax.Array, collections.abc.Sequence[Union[jax.Array, numpy.ndarray, numpy.bool_, numpy.number, bool, int, float, complex]]],
) -> jax.Array
Docstring:
Wraps XLA's `DynamicUpdateSlice
<https://www.tensorflow.org/xla/operation_semantics#dynamicupdateslice>`_
operator.

Args:
  operand: an array to slice.
  update: an array containing the new values to write onto `operand`.
  start_indices: a list of scalar indices, one per dimension.

Returns:
  An array containing the slice.

Examples:
  Here is an example of updating a one-dimensional slice update:

  >>> x = jnp.zeros(6)
  >>> y = jnp.ones(3)
  >>> dynamic_update_slice(x, y, (2,))
  Array([0., 0., 1., 1., 1., 0.], dtype=float32)

  If the update slice is too large to fit in the array, the start
  index will be adjusted to make 

In [ ]:
def g(x):
    return x[1:5]

jaxpr = jax.make_jaxpr(g)(jnp.ones((10,)))

jaxpr.eqns[0]



a:f32[4] = dynamic_slice[slice_sizes=(4,)] b 1

In [ ]:
ys = jax.core.eval_jaxpr(forward_jaxpr.jaxpr, forward_jaxpr.literals, 1.)
ys

[Array([1.   , 1.   , 1.   , 1.   , 1.001, 1.001, 1.002, 1.002, 1.003,
        1.004, 1.005, 1.006, 1.007, 1.008, 1.01 , 1.011, 1.013, 1.014,
        1.016, 1.018, 1.02 , 1.022, 1.024, 1.026, 1.029, 1.031, 1.034,
        1.036, 1.039, 1.042, 1.045, 1.048, 1.051, 1.054, 1.058, 1.061,
        1.064, 1.068, 1.072, 1.076, 1.08 , 1.084, 1.088, 1.092, 1.096,
        1.1  , 1.105, 1.109, 1.114, 1.119, 1.124, 1.129, 1.134, 1.139,
        1.144, 1.149, 1.154, 1.16 , 1.165, 1.171, 1.177, 1.182, 1.188,
        1.194, 1.2  , 1.206, 1.213, 1.219, 1.225, 1.232, 1.238, 1.245,
        1.251, 1.258, 1.265, 1.272, 1.279, 1.286, 1.293, 1.3  , 1.307,
        1.315, 1.322, 1.329, 1.337, 1.345, 1.352, 1.36 , 1.368, 1.376,
        1.384, 1.392, 1.4  , 1.408, 1.416, 1.424, 1.432, 1.441, 1.449,
        1.458], dtype=float32)]

In [ ]:
inverse_jaxpr.literals

[Array([0.   , 0.01 , 0.02 , 0.03 , 0.04 , 0.051, 0.061, 0.071, 0.081,
        0.091, 0.101, 0.111, 0.121, 0.131, 0.141, 0.152, 0.162, 0.172,
        0.182, 0.192, 0.202, 0.212, 0.222, 0.232, 0.242, 0.253, 0.263,
        0.273, 0.283, 0.293, 0.303, 0.313, 0.323, 0.333, 0.343, 0.354,
        0.364, 0.374, 0.384, 0.394, 0.404, 0.414, 0.424, 0.434, 0.444,
        0.455, 0.465, 0.475, 0.485, 0.495, 0.505, 0.515, 0.525, 0.535,
        0.545, 0.556, 0.566, 0.576, 0.586, 0.596, 0.606, 0.616, 0.626,
        0.636, 0.646, 0.657, 0.667, 0.677, 0.687, 0.697, 0.707, 0.717,
        0.727, 0.737, 0.747, 0.758, 0.768, 0.778, 0.788, 0.798, 0.808,
        0.818, 0.828, 0.838, 0.848, 0.859, 0.869, 0.879, 0.889, 0.899,
        0.909, 0.919, 0.929, 0.939, 0.949, 0.96 , 0.97 , 0.98 , 0.99 ,
        1.   ], dtype=float32),
 Array([[0. , 0. , 0. , 0. ],
        [0.5, 0. , 0. , 0. ],
        [0. , 0.5, 0. , 0. ],
        [0. , 0. , 1. , 0. ]], dtype=float32),
 Array([0.167, 0.333, 0.333, 0.167], dtype=float32

In [ ]:
jax.core.eval_jaxpr(inverse_jaxpr.jaxpr, inverse_jaxpr.literals,ys[0])

[Array(0.996, dtype=float32), Array(0., dtype=float32)]

In [252]:
import jax

import jax.numpy as jnp

jax.numpy.set_printoptions(precision=3, suppress=True)
from jax import core

from jax import linear_util as lu
from functools import partial, update_wrapper

from jax.tree_util import tree_flatten, tree_unflatten, tree_leaves, tree_map
from jax.interpreters import ad, batching
from jax._src import ad_util

from jax.core import Primitive, CallPrimitive
from jax._src.util import weakref_lru_cache, cache
from jax._src import util

from typing import Any, Callable
from jax._src.util import safe_map
from jax._src.api_util import (
    flatten_fun_nokwargs,
    argnums_partial,
    flatten_fun_nokwargs,
    shaped_abstractify,
)

from jax.interpreters import mlir
from jax.interpreters import partial_eval as pe

# This is a custom primitive that allows us to define custom inverse functions
# While most stuff can be inverted by inverting all primitives for some functions it is necessary or more efficient to define a custom inverse function

custom_inverse_call_p = Primitive("custom_inverse_call_p")
custom_inverse_call_p.multiple_results = True


@custom_inverse_call_p.def_impl
def custom_inverse_call_impl(*args, forward_jaxpr, inverse_jaxpr, **params):
    with core.new_sublevel():
        print(args)
        ans = core.eval_jaxpr(forward_jaxpr.jaxpr, forward_jaxpr.literals, *args)
        print(ans)
    return ans


@custom_inverse_call_p.def_abstract_eval
def custom_inverse_call_abstract_eval(*args, forward_jaxpr, inverse_jaxpr, **params):
    with core.new_sublevel():
        return forward_jaxpr.out_avals


def custom_inverse_call_lowering(ctx, *args, forward_jaxpr, inverse_jaxpr, **params):
    return mlir.core_call_lowering(
        ctx, *args, name="forward_call", call_jaxpr=forward_jaxpr
    )


mlir.register_lowering(custom_inverse_call_p, custom_inverse_call_lowering)

@jax.util.cache()
def process_jvp(forward_jaxpr, tangents):
    nonzeros = [type(t) is not ad_util.Zero for t in tangents]
    forward_jvp_jaxpr, forward_out_nz = ad.jvp_jaxpr(
        forward_jaxpr, nonzeros, instantiate=False
    )
    nonzero_tangents = [t for t in tangents if type(t) is not ad_util.Zero]
    forward_jvp_jaxpr_ = pe.convert_constvars_jaxpr(forward_jvp_jaxpr.jaxpr)
    return forward_jvp_jaxpr_,forward_jvp_jaxpr, nonzero_tangents

def custom_inverse_jvp(primals, tangents, forward_jaxpr, inverse_jaxpr, **params):
 
    forward_jvp_jaxpr_, forward_jvp_jaxpr, nonzero_tangents = process_jvp(forward_jaxpr, tangents)

    new_primals, new_tangent = core.eval_jaxpr(
        forward_jvp_jaxpr_, forward_jvp_jaxpr.consts, *primals, *nonzero_tangents
    )

    return [new_primals, ], [new_tangent, ]


def batch_custom_inverse_call(
    spmd_axis_name, axis_size, axis_name, main_type, args, dims, **params
):
    forward_jaxpr = params.pop("forward_jaxpr")
    inverse_jaxpr = params.pop("inverse_jaxpr")

    # We have to batch the jaxprs. For that lets first get the invals and outvals
    in_avals1 = forward_jaxpr.in_avals
    out_avals1 = forward_jaxpr.out_avals

    in_avals2 = inverse_jaxpr.in_avals
    out_avals2 = inverse_jaxpr.out_avals

    # We will batch all the inputs and outputs  (maybe do not batch consts ... )
    in_batched1 = [True] * len(in_avals1)
    out_batched1 = [True] * len(out_avals1)

    in_batched2 = [True] * len(in_avals2)
    out_batched2 = [True] * len(out_avals2)

    # Applies the batching for the jaxprs
    args = [batching.bdim_at_front(x, d, axis_size) for x, d in zip(args, dims)]

    # Batched jaxprs
    batched_forward_fn, out_size1 = batching.batch_jaxpr(
        forward_jaxpr,
        axis_size,
        in_batched1,
        out_batched1,
        axis_name,
        spmd_axis_name,
        main_type,
    )
    batched_inverse_fn, _ = batching.batch_jaxpr(
        inverse_jaxpr,
        axis_size,
        in_batched2,
        out_batched2,
        axis_name,
        spmd_axis_name,
        main_type,
    )

    # Update jaxprs with batched ones
    out = custom_inverse_call_p.bind(
        *args,
        forward_jaxpr=batched_forward_fn,
        inverse_jaxpr=batched_inverse_fn,
        **params,
    )

    # Outdim
    out_dims = [0 if b else batching.not_mapped for b in out_size1]

    return out, out_dims


def custom_inverse_transpose(*args, **kwargs):
    return ad.call_transpose(custom_inverse_call_p, *args, **kwargs)


batching.spmd_axis_primitive_batchers[custom_inverse_call_p] = batch_custom_inverse_call
batching.axis_primitive_batchers[custom_inverse_call_p] = partial(
    batch_custom_inverse_call, None
)
ad.primitive_transposes[custom_inverse_call_p] = custom_inverse_transpose
ad.primitive_jvps[custom_inverse_call_p] = custom_inverse_jvp


def is_hashable(obj):
    try:
        hash(obj)
        return True
    except TypeError:
        return False


# TODO: Add support other tracer support!
# TODO: Add support for caching! -> Otherwise we will have to retrace every time!




class custom_inverse:
    def __init__(self, fun: Callable, static_argnums=None) -> None:
        update_wrapper(self, fun)
        self.fun = fun
        self.static_argnums = static_argnums

    def definv(self, inv_fun: Callable) -> Callable:
        def wrapped_inv(*args, **kwargs):
            return inv_fun(*args, **kwargs), jnp.nan

        self.inv_fun = inv_fun
        self.inv_fun_and_log_det = wrapped_inv
        return wrapped_inv

    def definv_and_logdet(self, inv_fun_and_log_det: Callable) -> Callable:
        self.inv_fun_and_log_det = inv_fun_and_log_det
        if not hasattr(self, "inv_fun"):
            self.inv_fun = lambda *args, **kwargs: inv_fun_and_log_det(*args, **kwargs)[
                0
            ]
        return inv_fun_and_log_det

    def inv(self, *args, **kwargs):
        return self.inv_fun(*args, **kwargs)

    def inv_and_logdet(self, *args, **kwargs):
        return self.inv_fun_and_log_det(*args, **kwargs)

    def __call__(self, *args, **params) -> Any:
        name = getattr(self.fun, "__name__", str(self.fun))
        if not self.inv_fun:
            msg = f"No inverse defined for custom_inverse function {name} using definv."
            raise AttributeError(msg)
        inv_name = getattr(self.inv_fun, "__name__", str(self.inv_fun))

        # We can only invert with respect to specific dynamic arguments. All others are assumed to be static!
        f = lu.wrap_init(self.fun, params=params)
        f_inv = lu.wrap_init(self.inv_fun_and_log_det, params=params)

        # Dynamic and static args for forward and inverse
        if self.static_argnums is None:
            dyn_args = args
            dyn_args_index = None
        else:
            dyn_args_index = tuple([
                i
                for i in range(len(args))
                if i not in self.static_argnums  # or not is_hashable(args[i])
            ])

            f, dyn_args = argnums_partial(
                f, dyn_args_index, args, require_static_args_hashable=False
            )

            f_inv, _ = argnums_partial(
                f_inv, dyn_args_index, args, require_static_args_hashable=False
            )
 
        # Flatt stuff for tracing
        args_flat, in_tree = tree_flatten(dyn_args)
        in_avals = tuple(safe_map(shaped_abstractify, args_flat))

        forward_jaxpr, inverse_jaxpr, out_tree = trace_forward_inverse(f,f_inv,dyn_args_index,self.static_argnums, in_avals, in_tree, name, inv_name)

        out_flat = custom_inverse_call_p.bind(
            *args_flat,
            forward_jaxpr=forward_jaxpr,
            inverse_jaxpr=inverse_jaxpr,
            in_tree=in_tree,
        )

        return tree_unflatten(out_tree, out_flat)


In [253]:
@custom_inverse
def f(x):
    return x ** 2

def f_inv(x):
    return x ** 0.5

f.definv(f_inv)

<function __main__.custom_inverse.definv.<locals>.wrapped_inv(*args, **kwargs)>

In [257]:

def test_grad(x):
    x1 = f(x)
    x2 = f(x1)
    x3 = f(x2)
    x4 = f(x3)
    return x4


In [259]:

def test_grad(x):
    x1 = f.fun(x)
    x2 = f.fun(x1)
    x3 = f.fun(x2)
    x4 = f.fun(x3)
    return x4


In [256]:
%%timeit
test_grad(1.)

5.15 ms ± 686 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [260]:
%%timeit
jax.jit(jax.grad(lambda x: test_grad(x)))(2.)

30.9 ms ± 3.07 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [114]:
@jax._src.util.cache()
def trace_forward_inverse(f, f_inv,dyn_args_index,static_argnums, in_avals, in_tree, name, inv_name):
    # Forward 
    f, out_tree = flatten_fun_nokwargs(f, in_tree)  # type: ignore
    debug = pe.debug_info(f.f, in_tree, out_tree, False, name or "<unknown>")
    jaxpr, out_avals, consts = pe.trace_to_jaxpr_dynamic(
            f, in_avals, debug
    )
    forward_jaxpr = core.ClosedJaxpr(jaxpr, consts)
    out_tree = out_tree()

    # Inverse 
    f_inv, _ = flatten_fun_nokwargs(f_inv, in_tree)  # type: ignore

    if static_argnums is not None:
        inv_in_avals = [
                    in_avals[i] if i in static_argnums else out_avals[0]
                    for i in dyn_args_index
                ]
    else:
        inv_in_avals = in_avals
    
    jaxpr, _, consts = pe.trace_to_jaxpr_dynamic(f_inv, inv_in_avals, debug)
    inverse_jaxpr = core.ClosedJaxpr(jaxpr, consts)

    return forward_jaxpr, inverse_jaxpr, out_tree

    